# 19.1 A/B 测试设计 / A/B Test Design

**中文**：前面 18 个 Part 都在学"从数据里学到什么"。但数据科学最值钱的问题往往是**因果**的:*"改了这个按钮,用户真的会多点吗?"* "这个新算法,真的提升了留存吗?"——注意是"**改了 X 导致 Y 变了**",而不只是"X 和 Y 相关"。**相关不等于因果**是本部分(Part 19)的灵魂。而回答因果问题的**黄金标准**,就是 **A/B 测试(随机对照实验, RCT)**。本节讲**怎么设计**一个可信的 A/B 测试——这也是数据科学/产品面试的**超高频**考点。
**English**: The prior 18 parts were about "what can we learn from data." But data science's most valuable questions are often **causal**: *"if we change this button, will users really click more?"* "does this new algorithm actually improve retention?" — note "**changing X causes Y to change**," not just "X and Y are correlated." **Correlation ≠ causation** is the soul of this part (Part 19). And the **gold standard** for answering causal questions is the **A/B test (randomized controlled trial, RCT)**. This section covers **how to design** a trustworthy A/B test — a very frequent data-science/product interview topic.

---

**中文**：为什么 A/B 测试能证明因果？关键在**随机化(randomization)**。把用户**随机**分成两组:对照组(A,看旧版)、实验组(B,看新版)。随机分组保证两组在**所有其他因素**(年龄、活跃度、地区、你想到的和想不到的)上**统计上完全一样**——唯一的系统性差异就是"你施加的改动"。于是两组结果的差异,只能归因于这个改动。**随机化一举消灭了所有混杂(confounding)**——这是它的魔力。
**English**: Why can an A/B test prove causation? The key is **randomization**. Randomly split users into two groups: control (A, old version) and treatment (B, new version). Random assignment guarantees the two groups are **statistically identical on all other factors** (age, activity, region, everything you thought of and didn't) — the only systematic difference is "the change you applied." So any difference in outcomes can only be attributed to that change. **Randomization eliminates all confounding at once** — that is its magic.

**中文**：设计一个 A/B 测试,核心是回答:**"我需要多少样本(用户)、跑多久?"** 这由四个量的关系决定(**功效分析, power analysis**):
**English**: Designing an A/B test centers on: **"how many samples (users) do I need, and for how long?"** This is set by the relationship among four quantities (**power analysis**):
- **显著性水平 $\alpha$**(通常 0.05):**假阳性率**——本来没效果,你却误判"有效果"的概率(第一类错误)。
  **Significance level $\alpha$** (usually 0.05): the **false-positive rate** — probability of wrongly declaring an effect when there is none (Type I error).
- **功效 power $=1-\beta$**(通常 0.8):**真有效果时,你能检测出来的概率**($\beta$=第二类错误=假阴性率)。
  **Power $=1-\beta$** (usually 0.8): the probability of **detecting an effect when one truly exists** ($\beta$=Type II error=false-negative rate).
- **最小可检测效应 MDE(Minimum Detectable Effect)**:你**在意的最小提升**(如提升 2% 转化)。想检测越小的效应,需要越多样本。
  **Minimum Detectable Effect (MDE)**: the **smallest lift you care about** (e.g. +2% conversion). Detecting a smaller effect needs more samples.
- **样本量 $n$**:每组需要多少用户。

  **Sample size $n$**: how many users per group.

**中文**：这四者由一个公式绑定(两组比较均值):
**English**: These four are bound by one formula (comparing two group means):

$$n\ \approx\ \frac{2\,(z_{1-\alpha/2}+z_{1-\beta})^2\,\sigma^2}{\text{MDE}^2}$$

**中文**：直觉:想要**更小的 MDE**(检测更细微的效果)或**更高的功效**,就需要**平方级更多**的样本;数据**方差 $\sigma^2$ 越大**(噪声越大),也需要更多样本。这个公式是每个数据科学家上线实验前必算的。
**English**: Intuition: a **smaller MDE** (detect subtler effects) or **higher power** needs **quadratically more** samples; noisier data (larger $\sigma^2$) also needs more. Every data scientist computes this before launching an experiment.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 产品DS必考）**
> **中文**：A/B 测试=随机对照实验(RCT), **随机化消灭混杂**→能证因果(相关≠因果的解药)。四要素:**α**(假阳性,0.05)、**power=1−β**(检出真效果,0.8)、**MDE**(最小在意效应)、**n**(样本量), 由 $n\propto\sigma^2/\text{MDE}^2$ 绑定——MDE 减半→样本量×4。**上线前必做功效分析**算 n 和时长。**常见坑**:①**偷看(peeking)**——反复看结果提前叫停会暴涨假阳性(要用序贯检验或固定样本量);②**多重比较**(测很多指标→假阳性膨胀→Bonferroni/FDR校正);③**新奇效应/周内效应**(至少跑满整周);④**样本比失衡 SRM**(分流出bug);⑤**辛普森悖论**。指标:主指标(OEC)+护栏指标(guardrail)。
> **English**: A/B test = RCT; **randomization kills confounding** → proves causation (the cure for correlation≠causation). Four levers: **α** (false positive, 0.05), **power=1−β** (detect a real effect, 0.8), **MDE** (smallest effect you care about), **n** (sample size), bound by $n\propto\sigma^2/\text{MDE}^2$ — halve the MDE → 4× the sample. **Always run a power analysis** before launch to get n and duration. **Common pitfalls**: ① **peeking** — repeatedly checking and stopping early inflates false positives (use sequential tests or a fixed sample size); ② **multiple comparisons** (many metrics → false-positive inflation → Bonferroni/FDR); ③ **novelty / day-of-week effects** (run at least full weeks); ④ **sample-ratio mismatch (SRM)** (a splitter bug); ⑤ **Simpson's paradox**. Metrics: a primary metric (OEC) + guardrail metrics.


In [ ]:

# ============================================================
# 随机化为什么有效:模拟消灭混杂 / why randomization works: it eliminates confounding
# 中文:造一个"有混杂"的世界。用户有隐藏的"活跃度", 活跃用户更可能自己去用新功能(自选择),
#      活跃用户本身转化也高。若按"用没用新功能"分组(非随机)→ 会把"活跃度"的效应误算成"新功能"的效应。
# English: build a confounded world. Users have a hidden "activity"; active users self-select into the
#      new feature AND convert more. Grouping by "used the feature" (non-random) mixes activity into the effect.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.random.seed(0)
N=20000
activity=np.random.rand(N)                                    # 隐藏混杂:用户活跃度 / hidden confounder
true_effect=0.03                                              # 新功能的真实因果效应=+3个百分点 / true causal effect
# 基线转化率随活跃度上升 / baseline conversion rises with activity
base_p=0.10+0.30*activity

# 情形1:非随机(自选择)——活跃用户更爱用新功能 / self-selection: active users adopt more
used = (np.random.rand(N) < activity).astype(int)            # 用没用新功能, 与活跃度强相关 / correlated w/ confounder
conv_selfselect = (np.random.rand(N) < base_p + true_effect*used).astype(int)
naive_diff = conv_selfselect[used==1].mean() - conv_selfselect[used==0].mean()

# 情形2:随机化 A/B ——抛硬币分组, 与活跃度无关 / randomized: coin-flip assignment, independent of confounder
group = np.random.randint(0,2,N)                             # 随机分组 / random assignment
conv_ab = (np.random.rand(N) < base_p + true_effect*group).astype(int)
ab_diff = conv_ab[group==1].mean() - conv_ab[group==0].mean()

print(f"真实因果效应 / true causal effect: +{true_effect*100:.1f} 个百分点")
print(f"① 非随机(自选择)估计 / naive self-selected: +{naive_diff*100:.1f} pp  ← 被混杂严重高估!")
print(f"② 随机化 A/B 估计   / randomized A/B:      +{ab_diff*100:.1f} pp  ← 接近真值!")
print("随机化让两组活跃度均衡 / randomization balances the confounder:")
print(f"  自选择两组活跃度: 用了={activity[used==1].mean():.2f} vs 没用={activity[used==0].mean():.2f} (差很多)")
print(f"  随机两组活跃度:   B={activity[group==1].mean():.2f} vs A={activity[group==0].mean():.2f} (几乎相同)")


**中文**：看到了吗——**非随机分组把 +3% 的真实效应严重高估**(因为"用新功能的人"本来就更活跃、转化更高,这份功劳被错算给了新功能)。而**随机化 A/B** 让两组的活跃度几乎相同,估计出的效应接近真值。这就是"相关≠因果"最直接的演示,也是为什么我们需要随机实验。

接下来是设计的核心——**功效分析**:给定 MDE、α、power,算出需要多少样本。先用公式算,再用**模拟**验证这个样本量真能达到目标功效。
**English**: See it — **non-random grouping severely overestimates the true +3% effect** (because "people who used the feature" were already more active and higher-converting, and that credit is misattributed to the feature). **Randomized A/B** makes the two groups' activity nearly identical, and the estimate is close to the truth. This is the most direct demonstration of "correlation ≠ causation" and why we need randomized experiments.

Next, the core of design — **power analysis**: given MDE, α, power, compute the required sample size. First the formula, then verify by **simulation** that this sample size truly achieves the target power.


In [ ]:

# ============================================================
# 功效分析:算样本量 + 模拟验证 / power analysis: sample size formula + simulation check
# ============================================================
def sample_size(mde, sigma, alpha=0.05, power=0.8):
    z_a=stats.norm.ppf(1-alpha/2); z_b=stats.norm.ppf(power)  # 双侧α与功效的分位数 / z-quantiles
    return int(np.ceil(2*(z_a+z_b)**2*sigma**2/mde**2))       # 每组样本量公式 / per-group n

# 例:基线转化率 20%, 想检测 +2 个百分点(MDE=0.02)/ example: baseline 20%, MDE=+2pp
p0=0.20; mde=0.02; sigma=np.sqrt(p0*(1-p0))                   # 伯努利指标的标准差 / std of a rate metric
n=sample_size(mde, sigma)
print(f"基线 {p0:.0%}, MDE={mde:.0%}, α=0.05, power=0.8 → 每组需要 {n} 用户(共 {2*n})")

# 模拟验证:真效应=MDE 时, 反复跑实验, 看多少比例检测到显著(应≈power=0.8)
def empirical_power(n, p0, effect, alpha=0.05, trials=2000):
    detect=0
    for _ in range(trials):
        a=np.random.rand(n)<p0; b=np.random.rand(n)<(p0+effect)    # 两组样本 / two arms
        # 两比例 z 检验 / two-proportion z-test
        pa,pb=a.mean(),b.mean(); pp=(a.sum()+b.sum())/(2*n)
        se=np.sqrt(pp*(1-pp)*2/n); z=(pb-pa)/se
        if abs(z)>stats.norm.ppf(1-alpha/2): detect+=1
    return detect/trials
emp=empirical_power(n, p0, mde)
print(f"模拟验证:真效应=MDE 时检测到显著的比例 = {emp:.2f} (目标 power=0.80) → 公式正确!")


**中文**：可视化功效分析的三条核心关系:样本量 vs MDE(平方反比)、功效 vs 样本量(样本越多越能检测)、以及不同真实效应下的功效曲线。这些图是每个 A/B 平台"样本量计算器"背后的原理。
**English**: Visualize the three core relationships of power analysis: sample size vs MDE (inverse-square), power vs sample size (more samples = more detection), and power curves for different true effects. These plots are the principle behind every A/B platform's "sample size calculator."


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① 样本量 vs MDE (平方反比) / sample size vs MDE
mdes=np.linspace(0.005,0.05,50)
ns=[sample_size(m,sigma) for m in mdes]
ax[0].plot(mdes*100,ns,color="#4C72B0")
ax[0].set_title("样本量 vs MDE:检测越小效应, 样本平方级暴涨"); ax[0].set_xlabel("MDE (百分点)"); ax[0].set_ylabel("每组样本量 n"); ax[0].set_yscale("log")
ax[0].axvline(2,ls=":",color="r"); ax[0].annotate(f"MDE=2pp\n→ n={sample_size(0.02,sigma)}",(2,sample_size(0.02,sigma)),fontsize=8)
# ② 功效 vs 样本量 / power vs sample size
sizes=np.linspace(500,10000,20).astype(int)
powers=[empirical_power(s,p0,mde,trials=800) for s in sizes]
ax[1].plot(sizes,powers,"o-",color="#55A868")
ax[1].axhline(0.8,ls="--",color="gray",label="目标 power=0.8"); ax[1].axvline(n,ls=":",color="r",label=f"公式 n={n}")
ax[1].set_title("功效 vs 样本量 / power vs sample size"); ax[1].set_xlabel("每组样本量"); ax[1].set_ylabel("功效 power"); ax[1].legend(fontsize=8)
# ③ 不同真实效应下的功效曲线 / power curves for different true effects
for eff,c in [(0.01,"#C44E52"),(0.02,"#DD8452"),(0.04,"#4C72B0")]:
    pw=[empirical_power(s,p0,eff,trials=600) for s in sizes]
    ax[2].plot(sizes,pw,"o-",color=c,label=f"真效应=+{eff*100:.0f}pp")
ax[2].axhline(0.8,ls="--",color="gray"); ax[2].set_title("效应越大越容易检测 / bigger effect, easier to detect")
ax[2].set_xlabel("每组样本量"); ax[2].set_ylabel("功效"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/ci01_viz.png",dpi=80); plt.show()
print("MDE 减半→样本量×4; 效应越小/方差越大→需要越多样本, 这是实验设计的铁律")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **随机化是因果推断的"皇冠"**:非随机分组把 +3% 的真效应高估了一倍多(混杂在作祟),而随机化几乎无偏地还原了真值。**只要你能随机分组,A/B 测试就是最强、最可信的因果工具**——后面 19.5–19.8 讲的一堆复杂方法(匹配、DiD、IV、RDD),本质都是"**在不能随机化时,退而求其次地逼近随机实验**"。
2. **功效分析是上线前的必修课**:样本量 $\propto \sigma^2/\text{MDE}^2$——**MDE 减半,样本量翻 4 倍**。想检测越细微的提升,代价越是平方级暴涨。模拟验证了公式:真效应=MDE 时,检测到显著的比例正好≈80%(目标功效)。这解释了为什么大公司做小改动的 A/B 需要海量流量、跑很久。
3. **诚实的现实**:公式给的是"理想"样本量,真实实验还要考虑:①**不是所有指标方差都好估**(要用历史数据);②**用户不是独立的**(网络效应, 19.11);③**流量有限**——检测不到的小效应不代表没效应,只是**功效不够**("no evidence of effect" ≠ "evidence of no effect")。这是面试常考的微妙点。

**English**:
1. **Randomization is the crown of causal inference**: non-random grouping overestimated the +3% true effect by more than double (confounding at work), while randomization recovered the truth almost unbiasedly. **As long as you can randomize, an A/B test is the strongest, most trustworthy causal tool** — the complex methods in 19.5–19.8 (matching, DiD, IV, RDD) are essentially "**second-best approximations to a randomized experiment when you can't randomize**."
2. **Power analysis is mandatory before launch**: sample size $\propto \sigma^2/\text{MDE}^2$ — **halve the MDE, quadruple the sample**. Detecting subtler lifts costs quadratically more. Simulation verified the formula: when the true effect equals the MDE, the fraction detected as significant is exactly ≈80% (the target power). This is why big companies' A/B tests on small changes need enormous traffic over long durations.
3. **Honest reality**: the formula gives an "ideal" sample size; real experiments must also account for: ① **not all metric variances are easy to estimate** (use historical data); ② **users aren't independent** (network effects, 19.11); ③ **limited traffic** — failing to detect a small effect doesn't mean there is none, just **insufficient power** ("no evidence of effect" ≠ "evidence of no effect"). A subtle interview favorite.

> 💼 **实战视角 / Practical angle**
> **中文**:A/B 测试是**互联网产品决策的核心基础设施**(Google/Meta/字节每天跑上万个实验)。落地要点:①**先做功效分析**定样本量与时长(至少跑满整周避免周内效应);②**主指标(OEC)+护栏指标**(别为提点击牺牲留存/收入);③**别偷看**(要序贯检验或固定样本);④**多指标做校正**(FDR);⑤上线前查 **A/A 测试**(两组都是旧版, 应无差异——查分流是否公平)与 **SRM**;⑥效应异质(对不同人效果不同)→ 后面的 uplift/因果森林。面试金句:*"A/B 靠随机化消灭混杂来证因果; 设计核心是功效分析——样本量∝σ²/MDE², 上线前算 n 和时长; 最大的坑是偷看、多重比较、和网络效应违反 SUTVA。"*
> **English**: A/B testing is the **core decision infrastructure of internet products** (Google/Meta/ByteDance run tens of thousands of experiments daily). Deployment keys: ① **do a power analysis first** to set sample size and duration (run at least full weeks to avoid day-of-week effects); ② **a primary metric (OEC) + guardrails** (don't sacrifice retention/revenue for clicks); ③ **don't peek** (use sequential tests or a fixed sample); ④ **correct for many metrics** (FDR); ⑤ before launch run an **A/A test** (both arms identical, should show no difference — checks a fair split) and **SRM**; ⑥ heterogeneous effects (different effect per person) → uplift/causal forests later. Interview line: *"A/B proves causation by randomization killing confounding; the design core is power analysis — sample size ∝ σ²/MDE², compute n and duration before launch; the biggest pitfalls are peeking, multiple comparisons, and network effects violating SUTVA."*

---
### 小结 / Summary
- **中文**:A/B 测试=RCT, 随机化消灭混杂→证因果(相关≠因果的解药); 是因果推断的黄金标准。
- **English**: A/B test = RCT; randomization eliminates confounding → proves causation (the cure for correlation≠causation); the gold standard of causal inference.
- **中文**:功效分析绑定 α/power/MDE/n:$n\propto\sigma^2/\text{MDE}^2$, MDE 减半样本×4; 上线前必算。
- **English**: Power analysis binds α/power/MDE/n: $n\propto\sigma^2/\text{MDE}^2$; halve MDE → 4× sample; compute before launch.
- **中文**:核心坑:偷看、多重比较、周内效应、SRM、网络效应; 不显著≠无效应(功效不足)。
- **English**: Key pitfalls: peeking, multiple comparisons, day-of-week, SRM, network effects; non-significant ≠ no effect (insufficient power).
